In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [ ]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()
    
    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()
    
    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [ ]:
# MAIN (DEBUG)
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 0
# Load data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-10-31')

#Display
# print (asset_classes[asset_index],":\n")
# print(data)
with pd.option_context('display.max_columns', None,
                       'display.max_rows', None,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
    display(data.head(30))


In [ ]:
# Create a line plot using matplotlib

from matplotlib import colors


results_df = data.copy()

# Filter for first 30 days
first_date = results_df['TransactionDate'].min()
last_date = first_date + pd.Timedelta(days=30)
filtered_df = results_df[results_df['TransactionDate'] <= last_date]

plt.figure(figsize=(12, 7))
plt.step(filtered_df['TransactionDate'], filtered_df['Available'], where='post', label='Available $', color='#7B5028', linewidth=2)

# y-axis
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# x-axis (with adaptive frequency)
date_range = pd.date_range(first_date, last_date)
if len(date_range) > 15:
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=2))
else:
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%#m/%#d'))


# Rotate x-axis labels
plt.xticks(rotation=75)

# Labels and title
plt.xlabel('Date')
plt.ylabel('Available Amount ($)')
plt.title('Available $-amount over time for ' + asset_classes[asset_index], fontweight='bold')

# Add grid
plt.grid(True, linestyle='--', alpha=0.7)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Add value labels for each point
prev_y = None
fc = colors.to_rgba('white')
fc = fc[:-1] + (0.65,) # alpha 0.65

for x, y in zip(filtered_df['TransactionDate'], filtered_df['Available']):
    if y != prev_y:  # Only annotate if value is different from previous
        plt.annotate(f'${y/1e6:.1f}M', 
                (x, y),
                textcoords="offset points",
                xytext=(-5,5),
                ha='right',
                fontsize=8,
                rotation=290,
                backgroundcolor=fc
        )
        # Add markers at each data point
        plt.plot(filtered_df['TransactionDate'], filtered_df['Available'], 'o', color='#2E1E0F', markersize=4)
        prev_y = y
        # Add padding to the plot margins to prevent cutoff
        plt.margins(x=0.05, y=0.2)
plt.show()

In [ ]:
def find_minimum_available_intervals(df, threshold=1_000_000):
    """
    Find minimum available intervals - longest consecutive periods where at least X amount is available.
    
    For each unique amount in the data, finds the longest period where at least that amount
    was continuously available, recording the minimum amount during that period.
    
    Returns a dataframe with the same structure as find_balance_intervals().
    """
    start_dates = []
    end_dates = []
    amounts = []
    
    unique_amounts = sorted(df['Available'].unique())
    
    for target_amount in unique_amounts:
        if target_amount < threshold:
            continue
        
        # Find all consecutive periods where at least target_amount is available
        consecutive_periods = []
        current_start = None
        
        for i in range(len(df)):
            current_available = df.iloc[i]['Available']
            current_date = df.iloc[i]['TransactionDate']
            
            if current_available >= target_amount:
                if current_start is None:
                    current_start = current_date
            else:
                if current_start is not None:
                    # End of a consecutive period
                    end_date = df.iloc[i-1]['TransactionDate']
                    consecutive_periods.append((current_start, end_date))
                    current_start = None
        
        # Handle case where the last period extends to the end
        if current_start is not None:
            consecutive_periods.append((current_start, df.iloc[-1]['TransactionDate']))
        
        # Find the longest period for this target amount
        if consecutive_periods:
            longest_period = max(consecutive_periods, 
                               key=lambda x: (pd.to_datetime(x[1]) - pd.to_datetime(x[0])).days)
            
            longest_start, longest_end = longest_period
            
            # Find the minimum amount available during this longest period
            period_data = df[(df['TransactionDate'] >= longest_start) & 
                           (df['TransactionDate'] <= longest_end)]
            min_amount_in_period = period_data['Available'].min()
            
            start_dates.append(longest_start)
            end_dates.append(longest_end)
            amounts.append(min_amount_in_period)
    
    # Create result dataframe with same structure as original function
    intervals_df = pd.DataFrame({
        'StartDate': start_dates,
        'EndDate': end_dates,
        'Amount': amounts
    })
    
    # Sort by start date
    intervals_df = intervals_df.sort_values('StartDate').reset_index(drop=True)
    
    # Calculate duration in days
    intervals_df['Duration'] = (intervals_df['EndDate'] - intervals_df['StartDate']).dt.days + 1
    
    # Calculate amount change
    intervals_df['Difference'] = None # intervals_df['Amount'].diff()
       
    # Set the first row AmountChange to equal the Amount
    # if len(intervals_df) > 0:
    #     intervals_df.loc[0, 'Difference'] = intervals_df.loc[0, 'Amount']
    
    return intervals_df

In [ ]:
"""
df: DataFrame with 'TransactionDate' and 'Available' columns
threshold = 1,000,000 - minimum balance to consider

Returns intervals as follows:
- use the available balance for eachday
- derermine intervals for for how long the balance remains above a certain threshold of $1,000,000.
- So the result: 
- $16,860,124.55 is available from 9/4/2025 until the end
- $83,634,124.55 is available from 9/8/2025 until 9/9/2025
- $41,258,124.55 is available from 9/10/2025 until 9/24/2025
- and so on...
- add a column to the result dataframe with the duration of each interval in days
- add a column with the amount of how much more is available compared to the previous interval
"""

def find_balance_intervals(df, threshold=1_000_000):
    # Initialize lists to store interval data
    start_dates = []
    end_dates = []
    amounts = []
    
    # Initialize variables for tracking intervals
    current_amount = df.iloc[0]['Available']
    current_start = df.iloc[0]['TransactionDate']
    prev_date = df.iloc[0]['TransactionDate']
    
    # Iterate through the dataframe by date (day by day)
    for i in range(1, len(df)):
        current_row = df.iloc[i]
        
        # If amount changes or we reach the end
        if current_row['Available'] != current_amount or i == len(df) - 1:
            # If current amount is above threshold, save the interval
            if current_amount >= threshold:
                start_dates.append(current_start)
                end_dates.append(prev_date)
                amounts.append(current_amount)
            
            # Start new interval
            current_amount = current_row['Available']
            current_start = current_row['TransactionDate']
        
        prev_date = current_row['TransactionDate']
    
    # Create result dataframe
    intervals_df = pd.DataFrame({
        'StartDate': start_dates,
        'EndDate': end_dates,
        'Amount': amounts
    })
    
    # Calculate duration in days
    intervals_df['Duration'] = (intervals_df['EndDate'] - intervals_df['StartDate']).dt.days + 1
    
    # Calculate amount change
    intervals_df['Difference'] = intervals_df['Amount'].diff()
    # Set the first row AmountChange to equal the Amount
    intervals_df.loc[0, 'Difference'] = intervals_df.loc[0, 'Amount']

    # Drop rows where duration <= 1 day
    # intervals_df = intervals_df[intervals_df['Duration'] > 1].reset_index(drop=True)
    
    return intervals_df


In [ ]:
# Run the function on the data
# intervals = find_balance_intervals(data)
# display(intervals)

# Get exact intervals
exact_intervals = find_balance_intervals(data)

# print("Exact balance intervals:")
# display(exact_intervals)

# Get minimum available intervals  
min_intervals = find_minimum_available_intervals(data)

# difference compared to previous date's available balance
min_intervals['Difference'] = min_intervals.apply(
    lambda row: data[data['TransactionDate'] < row['StartDate']]['Available'].iloc[-1] 
    if not data[data['TransactionDate'] < row['StartDate']].empty 
    else row['Amount'], axis=1)
min_intervals['Difference'] = min_intervals['Amount'] - min_intervals['Difference']
min_intervals.loc[0, 'Difference'] = min_intervals.loc[0, 'Amount']

# print("Minimum available intervals:")
# display(min_intervals)

In [ ]:
# Get exact intervals
# exact_intervals = find_balance_intervals(df)

# Get minimum available intervals  
# min_intervals = find_minimum_available_intervals(df)

# Merge them if needed
exact_intervals['IntervalType'] = 'exact'
exact_intervals['IntervalType'] = exact_intervals['IntervalType'].astype('string')
min_intervals['IntervalType'] = 'minimum'
min_intervals['IntervalType'] = min_intervals['IntervalType'].astype('string')
combined = pd.concat([exact_intervals, min_intervals], ignore_index=True).sort_values('StartDate')
combined = combined.drop_duplicates(subset=['StartDate', 'EndDate', 'Amount'])
combined['AssetClass'] = asset_classes[asset_index]
# Reorder columns
combined = combined[['AssetClass', 'StartDate', 'EndDate', 'Amount', 'Duration', 'Difference', 'IntervalType']]

print("Available intervals combined:")
display(combined.sort_values(['StartDate','Duration']))



# Print column types for combined DataFrame
print("\nColumn Types:")
print(combined.dtypes)

# Print memory usage information
# print("\nMemory Usage:")
# print(combined.info())